# Bayesian optimization of group sequential designs

## Motivating problem

### Introduction

In Bayesian optimization, the goal is to optimize a blackbox function, $f$. In our case, this function takes inputs of 
$$
D = \{n, u_1, \ell_1, u_2, \ell_2, \cdots, u_k, \ell_k \}
$$

where 

- $D$ is the design (an $n$-tuple set),
- $n$ is the sample size at each analysis point (total for both groups),
- $u_1$ and $\ell_1$ are the upper and lower bounds for the first stage, and
- $k$ is the total number of stages

and outputs several values of interest, $T$ (an $m$-tuple set), including (but not limited to)

- $\alpha$, the type I error
- $\beta$, the type II error
- $\mathbb{E}[N \, | \, \boldsymbol{\delta}]$, the expected sample size (ESS) $N$ over a range of differences, $\delta$, between groups $\boldsymbol{\delta} = \{\delta_1, \delta_2, \dots, \delta_j \}$

Above, $N = k*n$ and 

The expected sample size is calculated across a set of possible true treatment effects $\boldsymbol{\delta} = \{\delta_1, \delta_2, \dots, \delta_j \}$ as a function of the design elements $D$: (1) number of analyses $\{1, 2, \dots, k\}$, (2) number of patients at each analysis $\mathbf{n} = \{n_1, n_2, \dots, n_k\}$, and (3) the upper and lower bounds $\mathbf{u} = (u_1, u_2, \dots, u_k)$ and $\boldsymbol{\ell} = (\ell_1, \ell_2, \dots, \ell_k)$.

It is calculated as
$$
\mathbb{E}[N \, | \, \boldsymbol{\delta}]=\sum_{i=1}^k n_i P(\text{trial stops after analysis }i \, | \, \boldsymbol{\delta})
$$

### Pseudocode for Bayesian optimization loop

In order to generated an optimized clinical trial design, the following steps must be followed:

1. Generate several feasible trial designs, $\boldsymbol{D}=\{D_1, D_2, \cdots, D_p\}$.
2. Generate the corresponding outputs, $\boldsymbol{T}=\{T_1, T_2, \cdots, T_p\}$.
3. Fit a Gaussian process regression model to estimate the blackbox function $f: D \to T$.
4. Perform a step of Bayesian optimization to find the next trial design of interest $D_i$.
5. Obtain the corresponding outputs that correspond to this design $T_i$.
6. Refit the Gaussian process regression model on the new data $n$-tuples $\{(\boldsymbol{D}, \boldsymbol{T}), (D_i, T_i)\}$

Repeat until termination policy is reached.

### Function to minimize

As the above optimization problem applies to clinical trial designs, there are certain feasibility constraints that must be considered. Most importantly, the type I and type II error (or power)&mdash;$\alpha$, $\beta$ (or $1-\beta$),respectively&mdash;must be near the set nominal levels. In other words, if the design requires that a one-sided $\alpha = 0.025$, then feasible designs must have values of $\alpha$ near this value. In Wason et al. (Statist. Med. 2012, 31 301–312), feasible designs are defined as "design[s] for which the significance level and power meet the required constraints."

Though Bayesian optimization can be constrained in such a manner, for simplicity, a penalty term will be included within the objective function $f$, which will take these constraints into account. Again, borrowing from Wason et al., the penalty term is:

$$
\mathcal{L} = \mu \cdot \left( \mathbb{I}_{\{\alpha' > \alpha\}}\cdot\frac{\alpha' - \alpha}{\alpha} + \mathbb{I}_{\{\beta' > \beta\}}\cdot\frac{\beta' - \beta}{\beta}  \right)
$$

where $\mu$ is the sample size for a one-stage design, $\alpha$ and $\beta$ are the set nominal values for type I and II error, respectively, $\alpha'$ and $\beta'$ are the type I and II errors for the new design, and $\mathbb{I}$ is the indicator function.

The function that we aim to minimise, $f$, is the sum of the maximum expected sample size of the design and a penalty function that penalizes designs that are not considered feasible:

$$
f = \max\left\{\mathbb{E}[N \, | \, \boldsymbol{\delta}]\right\} + \mathcal{L}
$$

## Implementation of Bayesian optimization loop

In [1]:
# higher resolution graphs
%config InlineBackend.figure_format='retina'

### Step 1: Generate study designs

In [2]:
# imports for study design step (Step 1)
import numpy as np
import pandas as pd
from scipy import stats

# imports for GP regression (Step 3)
import gpflow

# imports for Bayes opt (Step 4-6)
import trieste
from trieste.space import Box
from trieste.models.gpflow.models import GaussianProcessRegression
import tensorflow as tf

/usr/local/lib/python3.11/dist-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
/usr/local/lib/python3.11/dist-packages/gpflow/versions.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Test speed of multivariate integration vs. single variable integration

In [3]:
mvn = stats.multivariate_normal(mean = [1, 2], cov = [[1, 0.5], [0.5, 1]])

In [4]:
mvn.cdf([2, np.inf], lower_limit=[-np.inf, -np.inf])

0.8413447460685429

In [5]:
stats.norm.cdf(2, 1, 1)

0.8413447460685429

In [6]:
%%timeit
stats.norm.cdf(2, 1, 1)

17.4 μs ± 446 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [7]:
%%timeit
mvn.cdf([2, np.inf], lower_limit=[-np.inf, -np.inf])

17.6 μs ± 280 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


#### Simulate group sequential designs function

In [8]:
# simulate the trials to obtain alpha and beta
def simulate_group_sequential_designs(
        n_analyses=3,
        upper_bounds=[2.5, 2, 1.5],
        lower_bounds=[0, 0.75, 1.5],
        n_patients=20,
        null_hypothesis=0,
        alt_hypothesis=0.5,
        variance=1):

    # convert list to np ndarray
    upper_bounds = np.array(upper_bounds)
    lower_bounds = np.array(lower_bounds)
    
    # assign values for null and alt hypotheses
    theta_0 = null_hypothesis
    delta = alt_hypothesis

    # empty list to fill mean vectors
    mean_0 = []
    mean_1 = []

    # number of patients in each analysis
    n_patients_analysis = np.array([x for x in range(1, n_analyses + 1, 1)]) * n_patients

    # need to parse the upper and lower boundaries of the design
    # for futility and efficacy, must put the bounds of integration correctly
    # for pmvnorm
    futility_l_bounds = [[]]
    futility_u_bounds = [[]]
    efficacy_l_bounds = [[]]
    efficacy_u_bounds = [[]]

    n_analyses = len(upper_bounds)

    # loop through number of analyses
    for i in range(n_analyses):

        # special case of i = 1
        if i == 0:
            futility_l_bounds[i].append(-np.inf)
            futility_u_bounds[i].append(lower_bounds[i])
            efficacy_l_bounds[i].append(upper_bounds[i])
            efficacy_u_bounds[i].append(np.inf)
            continue

        # all other cases
        futility_l_bounds.append(np.append(lower_bounds[0:i], -np.inf))
        futility_u_bounds.append(np.append(upper_bounds[0:i], lower_bounds[i]))
        efficacy_l_bounds.append(np.append(lower_bounds[0:i], upper_bounds[i]))
        efficacy_u_bounds.append(np.append(upper_bounds[0:i], np.inf))

    # empty dictionary of SIGMA matrices
    SIGMA_dict = dict()

    # generate the SIGMA matrices
    for i in range(n_analyses):

        if i == 0: SIGMA_dict.update({i : np.sqrt(variance)})

        # start with diagonal matrix for SIGMA
        SIGMA = np.eye(N = i+1)

        # n = 2, need to fill all but 11, 22
        # n = 3, need to fill all but 11, 22, 33
        # n = 4, need to fill all but 11, 22, 33, 44
        # etc.
        for j in range(i+1):
            for k in range(i+1):

                # leave the 1s on the diagonal, skip interation
                if j == k: continue

                # when j is less than k, the lower number of patients will be in numerator
                if j < k: SIGMA[j,k] = np.sqrt(n_patients_analysis[j] / n_patients_analysis[k])

                # when j is greater than j, the lower number of patients will be in numerator
                if j > k: SIGMA[j,k] = np.sqrt(n_patients_analysis[k] / n_patients_analysis[j])

        SIGMA_dict.update({i : SIGMA})

    # empty data frame to collect probabilities
    # dictionary to DataFrame, column is the key:value pair of the dictionary
    probs_to_return = {
        "futility_null" : np.empty(n_analyses),
        "efficacy_null" : np.empty(n_analyses),
        "futility_alt" : np.empty(n_analyses),
        "efficacy_alt" : np.empty(n_analyses)
    }

    # generate the row names
    row_names = ["analysis_" + str(row_names) for row_names in range(1, n_analyses + 1, 1)]

    # create the empty data frame with data and index
    probs_to_return = pd.DataFrame(data = probs_to_return, index = row_names)
    
    # start calculations for the analyses
    for i in range(n_analyses):
        
        # mean under null
        mean_0.append(theta_0 * np.sqrt(n_patients_analysis[i] / (2 * variance)))

        # mean under alternative
        mean_1.append(delta   * np.sqrt(n_patients_analysis[i] / (2 * variance)))
        
        # generate the null and alt multivariate normal
        mvn_null = stats.multivariate_normal(mean = mean_0, cov = SIGMA_dict[i])
        mvn_alt  = stats.multivariate_normal(mean = mean_1, cov = SIGMA_dict[i])
        
        # prob stop for futility under null
        futility_null = mvn_null.cdf(futility_u_bounds[i], lower_limit = futility_l_bounds[i])
        
        # prob stop for futility under alt
        futility_alt = mvn_alt.cdf(futility_u_bounds[i], lower_limit = futility_l_bounds[i])
        
        # prob stop for efficacy under null
        efficacy_null = mvn_null.cdf(efficacy_u_bounds[i], lower_limit = efficacy_l_bounds[i])

        # prob stop for efficacy under alt
        efficacy_alt = mvn_alt.cdf(efficacy_u_bounds[i], lower_limit = efficacy_l_bounds[i])

        # add them to the data frame to return
        probs_to_return.iloc[i] = [futility_null, efficacy_null, futility_alt, efficacy_alt]

    # get the type I error (alpha)
    alpha = probs_to_return.sum(axis = 0)["efficacy_null"]

    # get the power (1 - beta)
    power = probs_to_return.sum(axis = 0)["efficacy_alt"]

    # get expected sample size
    summed_probs = probs_to_return.iloc[:,2:4].sum(axis=1)
    expected_sample_size = np.sum(summed_probs * n_patients_analysis)
    
    return probs_to_return, alpha, power, expected_sample_size
        

# Are the diagonals of the covariance matrix supposed to be the variance? Or are they always 1?

In [9]:
simulate_group_sequential_designs(
    n_analyses=3,
    upper_bounds=[2.5, 2, 1.5],
    lower_bounds=[0, 0.75, 1.5],
    n_patients=[20],
    null_hypothesis=0,
    alt_hypothesis=0.5,
    variance=1
)

(            futility_null  efficacy_null  futility_alt  efficacy_alt
 analysis_1       0.500000       0.006210      0.056923      0.179084
 analysis_2       0.298776       0.019415      0.042323      0.419684
 analysis_3       0.137286       0.038313      0.049028      0.252957,
 0.06393770302459909,
 0.8517252850396675,
 41.31957428970111)

#### Functions for 3 different study designs.

#### Pocock boundaries function

In [10]:
# Pocock boundaries
def calculate_pocock_boundaries(
        n_analyses=3,
        alpha=0.05,
        n_patients=20,
        sided="one.sided"):
    
    # the precision of the estimate for alpha
    epsilon = 1e-8

    # starting first set of bounds
    ub1 = np.repeat(0, repeats = n_analyses)
    lb1 = -np.repeat(0, repeats = n_analyses)

    # starting second set of bounds
    ub2 = np.repeat(10, repeats = n_analyses)
    lb2 = -np.repeat(10, repeats = n_analyses)

    # first alpha calcuation
    _, sim_alpha, _, _ = simulate_group_sequential_designs(
        n_analyses = n_analyses,
        upper_bounds = ub1,
        lower_bounds = lb1,
        n_patients = n_patients
    )

    while abs(sim_alpha - alpha) > epsilon:

        # calculate the first midpoint
        mid_u = (ub1 + ub2) / 2
        mid_l = -mid_u

        # calculate the simulated alpha
        probs, sim_alpha, power, ess = simulate_group_sequential_designs(
            n_analyses = n_analyses,
            upper_bounds = mid_u,
            lower_bounds = mid_l,
            n_patients = n_patients
        )

        if sim_alpha > alpha:
            ub1 = mid_u
            lb1 = mid_l
        else:
            ub2 = mid_u
            lb2 = mid_l

    if sided == "one.sided":
        return [
            ub1,
            np.append(lb1[0:n_analyses-1], ub1[n_analyses-1]),
            probs,
            sim_alpha,
            power,
            ess
        ]
    else:
        return [
            ub1,
            lb1,
            probs,
            sim_alpha,
            power,
            ess
        ]

In [11]:
calculate_pocock_boundaries()

[array([1.99218631, 1.99218631, 1.99218631]),
 array([-1.99218631, -1.99218631,  1.99218631]),
             futility_null  efficacy_null  futility_alt  efficacy_alt
 analysis_1       0.023175       0.023175  1.762383e-04      0.340519
 analysis_2       0.015488       0.015488  8.542716e-06      0.290513
 analysis_3       0.011337       0.011337  6.418981e-07      0.177920,
 0.049999992429021284,
 0.8089526896165236,
 29.110045778227718]

#### O'Brien-Fleming boundaries function

In [12]:
# O'Brien-Fleming boundaries
def calculate_of_boundaries(
        n_analyses=3,
        alpha=0.05,
        n_patients=20,
        sided="one.sided"):

    # the precision of the estimate for alpha
    epsilon = 1e-8

    # starting first set of bounds
    ub1 = np.repeat(0, repeats = n_analyses)
    lb1 = -np.repeat(0, repeats = n_analyses)

    # starting second set of bounds
    ub2 = np.repeat(10, repeats = n_analyses)
    lb2 = -np.repeat(10, repeats = n_analyses)

    # of bounds
    of_u1 = ub1 * (np.arange(1, n_analyses+1, 1)/n_analyses)**(-0.5)
    of_l1 = -of_u1
    
    # first alpha calcuation
    _, sim_alpha, _, _ = simulate_group_sequential_designs(
        n_analyses = n_analyses,
        upper_bounds = of_u1,
        lower_bounds = of_l1,
        n_patients = n_patients
    )

    while abs(sim_alpha - alpha) > epsilon:

        # calculate the first midpoint
        mid_u = (ub1 + ub2) / 2
        mid_l = -mid_u

        # convert to O'Brien-Fleming bounds
        mid_of_u = mid_u * (np.arange(1, n_analyses+1, 1)/n_analyses)**(-0.5)
        mid_of_l = -mid_of_u

        # calculate the simulated alpha
        probs, sim_alpha, power, ess = simulate_group_sequential_designs(
            n_analyses = n_analyses,
            upper_bounds = mid_of_u,
            lower_bounds = mid_of_l,
            n_patients = n_patients
        )

        if sim_alpha > alpha:
            ub1 = mid_u
            lb1 = mid_l
        else:
            ub2 = mid_u
            lb1 = mid_l

    if sided == "one.sided":
        return [
            mid_of_u,
            np.append(mid_of_l[0:n_analyses-1], mid_of_u[n_analyses-1]),
            probs,
            sim_alpha,
            power,
            ess
        ]
    else:
        return [
            mid_of_u,
            mid_of_l,
            probs,
            sim_alpha,
            power,
            ess
        ]

In [13]:
calculate_of_boundaries()

[array([2.9611209 , 2.09382866, 1.70960395]),
 array([-2.9611209 , -2.09382866,  1.70960395]),
             futility_null  efficacy_null  futility_alt  efficacy_alt
 analysis_1       0.001533       0.001533      0.000003      0.083796
 analysis_2       0.017207       0.017207      0.000007      0.475045
 analysis_3       0.031260       0.031261      0.000004      0.296344,
 0.050000004889945715,
 0.8551855963165168,
 38.45894350693506]

#### Triangular boundaries function

In [14]:
# Triangular boundaries
def calculate_triangular_boundaries(
        n_analyses = 3,
        alpha = 0.05,
        delta = 0.5):

    # maximum information calculation
    I_L_term1 = (4 * (0.583**2))/n_analyses
    I_L_term2 = 8 * np.log(1/(2*alpha))
    I_L_term3 = (2*0.583) / np.sqrt(n_analyses)

    I_L = (np.sqrt(I_L_term1 + I_L_term2) - I_L_term3)**2 * (1/delta)**2

    # boundary calculation
    bounds_term1 = (2/delta) * np.log(1/(2*alpha))
    bounds_term2 = 0.583 * np.sqrt(I_L/n_analyses)

    analysis_fracs = np.arange(1, n_analyses+1, 1)/n_analyses

    I_L_fracs = I_L * analysis_fracs

    e_l = (bounds_term1 - bounds_term2 + ((0.25*delta) * analysis_fracs * I_L ))/np.sqrt(I_L_fracs)

    f_l = (-bounds_term1 + bounds_term2 + ((0.75*delta) * analysis_fracs * I_L ))/np.sqrt(I_L_fracs)

    return [
        e_l,
        f_l, 
        I_L_fracs
    ]

In [15]:
calculate_triangular_boundaries()

[array([2.11957748, 1.87345951, 1.83560794]),
 array([6.28553399e-16, 1.12407571e+00, 1.83560794e+00]),
 array([17.97043475, 35.9408695 , 53.91130426])]

### Step 2: Simulate the trials to obtain $\alpha$, $\beta$, maximum ESS

There are two sets of variables important in determining maximum expected sample size: (1) the difference, $\delta$, of interest between the groups of the trial (e.g., null hypothesis $\delta_0=0$ and alternative hypothesis $\delta_1=0.5$ implies a difference of interest of 0.5 between the two groups) and (2) the design of the trial (i.e., the selected type I error rate, $\alpha$, type II error rate, $\beta$, and boundaries). Though the boundaries determine the type I error rate, $\alpha$, there is an infinite set of boundaries that meet this condition.

There are further dependencies that determine the type II error rate, $\beta$, namely the sample size for the trial. Typically, the larger the sample size, the smaller the type II error rate (or conversely, the higher the power, $1-\beta$). The variance of the selected measure also affects the type II error rate. As the variance increases, there will be a smaller standardized effect size ($\frac{\delta}{\sigma^2}$, and therefore a larger sample size will be required to obtain the same $\beta$.

![Maximum expected sample size dependencies](maximum-expected-sample-size-dependencies.png)

Because of these dependencies, Wason et al. 2011 fix the following:
- Type I error rate: $\alpha=0.05$
- Type II error rate: $\beta=0.1$
- Null difference: $\delta_0=0$
- Alternative difference: $\delta_1=1$
    - Implying difference of interest: $\delta=1$
- Standard deviation: $\sigma=3$ (i.e., variance, $\sigma^2=9$)

# Why does the single stage design sample size vary from expected calcuation?

#### Sample size calculator function

In [16]:
# sample size per group for a difference in means
def sample_size_means(
        ratio=1,
        variance=1,
        power=0.8,
        alpha=0.05,
        delta=1):

    # ratio of smaller group to larger group
    r = (ratio+1)/ratio

    # z statistic for power
    z_power = stats.norm.ppf(power)
    
    # z statistic for alpha
    z_alpha = abs(stats.norm.ppf(alpha/2))

    # sample size
    n = r * ((variance**2 * (z_power+z_alpha)**2) / delta**2)

    return n

In [17]:
sample_size_means(variance=3, power=0.9)

189.1336151059312

#### Maximum expected sample size function

Create a function that calculates the maximum expected sample size using interval bisection.

# How do I make sure that the search space (`delta_start` and `delta_stop`) include the maximum sample size?

In [18]:
# obtain maximum expected sample size for an interval of interest
def max_ess(
        delta_start=0,
        delta_stop=3,
        n_analyses=3,
        upper_bounds=[2.5, 2, 1.5],
        lower_bounds=[0, 0.75, 1.5],
        n_patients=20,
        null_hypothesis=0,
        variance=1):

    # epsilon precision for max ESS calculated
    epsilon = 1e-4

    # random starting values of ess
    ess_delta_start = 10
    ess_delta_stop = 0
    
    # while the error is greater than desired precision
    while abs(ess_delta_start - ess_delta_stop) > epsilon:

        # simulate the trial under delta_start
        probs, sim_alpha, power, ess_delta_start = simulate_group_sequential_designs(
            n_analyses = n_analyses,
            upper_bounds = upper_bounds,
            lower_bounds = lower_bounds,
            n_patients = n_patients,
            null_hypothesis = null_hypothesis,
            alt_hypothesis = delta_start,
            variance = variance
        )
    
        # simulate trial under delta_stop
        probs, sim_alpha, power, ess_delta_stop = simulate_group_sequential_designs(
            n_analyses = n_analyses,
            upper_bounds = upper_bounds,
            lower_bounds = lower_bounds,
            n_patients = n_patients,
            null_hypothesis = null_hypothesis,
            alt_hypothesis = delta_stop,
            variance = variance
        )
        
        if ess_delta_start >= ess_delta_stop:
            delta_stop = (delta_start + delta_stop)/2 
        else:
            delta_start = (delta_start + delta_stop)/2

    return ess_delta_start

In [19]:
max_ess()

43.8343886040859

Check to make sure this is the correct answer based on a grid search.

In [20]:
deltas=np.linspace(start=0, stop=3, num=1000)
ess_list = np.empty(1000)

for delta in deltas:
    probs, sim_alpha, power, ess = simulate_group_sequential_designs(
        alt_hypothesis = delta
    )

    ess_list = np.append(ess_list, ess)
    
ess_list.max()

2.3055478344297936e+286

#### Sample size finder function

In [21]:
def find_sample_size(
        power_target=0.9,
        n_analyses=3,
        upper_bounds=[2.5, 2, 1.5],
        lower_bounds=[0, 0.75, 1.5],
        null_hypothesis=0,
        alt_hypothesis=0.5,
        variance=1):

    # precision to get to power
    epsilon = 1e-8

    # initial sample sizes
    n_patients_left=1
    n_patients_right=1e6

    # calcuate first power
    probs, sim_alpha, power, ess = simulate_group_sequential_designs(
        n_analyses = n_analyses,
        upper_bounds = upper_bounds,
        lower_bounds = lower_bounds,
        n_patients = n_patients_left,
        null_hypothesis = null_hypothesis,
        alt_hypothesis = alt_hypothesis,
        variance = variance
    )
    
    # interval bisection loop
    while abs(power - power_target) > epsilon:

        # generate the midpoint
        n_patients_mid = (n_patients_left + n_patients_right)/2

        # calcuate power
        probs, sim_alpha, power, ess = simulate_group_sequential_designs(
            n_analyses = n_analyses,
            upper_bounds = upper_bounds,
            lower_bounds = lower_bounds,
            n_patients = n_patients_mid,
            null_hypothesis = null_hypothesis,
            alt_hypothesis = alt_hypothesis,
            variance = variance
        )

        if power > power_target:
            n_patients_right = n_patients_mid
        else:
            n_patients_left = n_patients_mid

    return [n_patients_left, power]

In [22]:
find_sample_size(power_target=0.7)

[12.5289197527145, 0.6999999902311552]

#### Feasibility penalty function

In [23]:
# generate the penalty term
def feasibility_penalty(
        ratio=1,
        variance=1,
        power=0.8,
        alpha=0.05,
        delta=1,
        beta_prime=0.7,
        alpha_prime=0.1):

    # calculate the sample size for one-stage design
    mu = sample_size_means(
        ratio=ratio,
        variance=variance,
        power=power,
        alpha=alpha,
        delta=delta
    )

    # create the indicator functions
    def alpha_indicator(alpha_prime):
        if alpha_prime > alpha: return 1
        return 0

    def beta_indicator(beta_prime):
        if beta_prime > power: return 1
        return 0

    return mu * ( (((alpha_prime-alpha)/alpha)*alpha_indicator(alpha_prime)) + (((beta_prime-power)/power)*beta_indicator(beta_prime)) )

In [24]:
feasibility_penalty()

15.697759468698182

### Step 3: Fit a Gaussian process regression model

First, we need to create a function that will output the single value to minimize.

In [25]:
# function to minimize
def function_to_minimize(max_ess_val, penalty):

    return max_ess_val + penalty

#### What are the inputs and outputs for GP regression and Bayes opt?

**Fixed inputs:**
- Type I error rate: $\alpha=0.05$
- Type II error rate: $\beta=0.1$
- Null difference: $\delta_0=0$
- Alternative difference: $\delta_1=1$
    - Implying difference of interest: $\delta=1$
- Standard deviation: $\sigma=3$ (i.e., variance, $\sigma^2=9$)

**Variable inputs:**
- Maximum sample size
- Feasibility penalty
- Boundary values
- Number of analyses, $k$

**Outputs**:
- New boundary values (from Bayes opt)
- New $\alpha := \alpha'$
- New $\beta := \beta'$

#### Pseudocode

1. Start with feasible trial design boundaries.
2. Using the fixed inputs, generate the maximum expected sample size and feasibility penalty.
3. Calculate the function value, $f$, using the maximum expected sample size and feasibility penalty.
4. Into the Gaussian process (GP) regression model, the design boundaries are inputs and the function values are outputs.
5. Using the GP regression model, perform one iteration of Bayesian optimization to find the next input values for the Gaussian process.
6. Using the result of Bayesian optimization (new boundary values), calculate:
    - $\alpha'$
    - $\beta'$
    - maximum expected sample size
    - feasibility penalty
    - new function value $f$
7. Feed the input (boundary values) and output (function value) into the GP regression model.
8. Obtain new design boundaries.
9. Repeat until termination policy reached.

#### Input dimensions

In order to fit the multidimensional Gaussian process regression, it is important to know how many input dimensions exist for the problem at hand. For a one-sided statistical test, there will be $2k-1$ unique boundaries (as the last boundary value is equal) and a sample size scaled by each stage (e.g., $n=20$ with $k=3$ stages implies $n_1=20, n_2=40, n_3=60$) for a total of $2k$ inputs.

In [26]:
# define a function that generates the input
# takes boundaries and n_patients as inputs and outputs a single array
# may need to add scaling if GPR fit is challenging
def generate_gpr_input(
        n_analyses,
        upper_bounds,
        lower_bounds,
        n_patients):

    # turn n_patients into an array
    n_patients = np.array([n_patients])

    # remove the redundant value from the lower bound
    lower_bounds_trunc = lower_bounds[0:n_analyses-1]

    # concatenate and return
    return np.concatenate((upper_bounds, lower_bounds_trunc, n_patients))

# The maximum expected sample size is at a different delta than the alternative for finding the correct beta?

In [27]:
################
# GPR workflow #
################

# some set defaults
num_analyses = 3
target_alpha = 0.05
target_power = 0.9
important_diff_delta = 1
assumed_variance = 3

# to obtain mu (sample size at one stage)
group_ratio = 1

# simulate the trial design 
simulation = calculate_pocock_boundaries(
    n_analyses=num_analyses,
    alpha=target_alpha,
    n_patients=20
)

# pull inputs to the below functions from simulation
upper = simulation[0]
lower = simulation[1]
alpha_prime = simulation[3]

# 1. Find the number of patients that achieves 90% power (beta 0.1)
# here we get beta_prime and alpha prime
n_power09, beta_prime = find_sample_size(n_analyses=num_analyses,
                                         upper_bounds=upper,
                                         lower_bounds=lower,
                                         alt_hypothesis=important_diff_delta,
                                         variance=assumed_variance)

# 2. Generate the GPR input values
# note that the input includes the sample size at power 0.9
x1 = generate_gpr_input(n_analyses = num_analyses,
                       upper_bounds=upper,
                       lower_bounds=lower,
                       n_patients=n_power09)

# 3. Generate maximum expected sample size and feasibility penalty
max_ess_new = max_ess(n_analyses=num_analyses,
                      upper_bounds=upper,
                      lower_bounds=lower,
                      n_patients=n_power09)

penalty = feasibility_penalty(ratio=group_ratio,
                              variance=assumed_variance,
                              power=target_power,
                              alpha=target_alpha,
                              delta=important_diff_delta,
                              beta_prime=beta_prime,
                              alpha_prime=alpha_prime)


# 4. Calculate the function value (GPR output)
y1 = function_to_minimize(max_ess_val=max_ess_new, penalty=penalty)

In [28]:
# simulate the trial design 
simulation = calculate_of_boundaries(
    n_analyses=num_analyses,
    alpha=target_alpha,
    n_patients=20
)

# pull inputs to the below functions from simulation
upper = simulation[0]
lower = simulation[1]
alpha_prime = simulation[3]

# 1. Find the number of patients that achieves 90% power (beta 0.1)
# here we get beta_prime and alpha prime
n_power09, beta_prime = find_sample_size(n_analyses=num_analyses,
                                         upper_bounds=upper,
                                         lower_bounds=lower,
                                         alt_hypothesis=important_diff_delta,
                                         variance=assumed_variance)

# 2. Generate the GPR input values
# note that the input includes the sample size at power 0.9
x2 = generate_gpr_input(n_analyses = num_analyses,
                       upper_bounds=upper,
                       lower_bounds=lower,
                       n_patients=n_power09)

# 3. Generate maximum expected sample size and feasibility penalty
max_ess_new = max_ess(n_analyses=num_analyses,
                      upper_bounds=upper,
                      lower_bounds=lower,
                      n_patients=n_power09)

penalty = feasibility_penalty(ratio=group_ratio,
                              variance=assumed_variance,
                              power=target_power,
                              alpha=target_alpha,
                              delta=important_diff_delta,
                              beta_prime=beta_prime,
                              alpha_prime=alpha_prime)


# 4. Calculate the function value (GPR output)
y2 = function_to_minimize(max_ess_val=max_ess_new, penalty=penalty)

In [29]:
x1

array([ 1.99218506,  1.99218506,  1.99218506, -1.99218506, -1.99218506,
       19.96291833])

In [30]:
x2

array([ 2.96112293,  2.09383011,  1.70960512, -2.96112293, -2.09383011,
       17.55406804])

In [31]:
y1

57.41980740442281

In [32]:
y2

51.95061152254732

In [33]:
X = np.concatenate((x1,x2)).reshape(2,6)

In [34]:
X

array([[ 1.99218506,  1.99218506,  1.99218506, -1.99218506, -1.99218506,
        19.96291833],
       [ 2.96112293,  2.09383011,  1.70960512, -2.96112293, -2.09383011,
        17.55406804]])

In [35]:
Y = np.concatenate(([y1], [y2])).reshape(2,1)

In [36]:
Y

array([[57.4198074 ],
       [51.95061152]])

# The above loop finds the boundaries then adjusts the sample size to obtain the desired power. In the paper, the boundaries are optimized to obtain the target power without modifying sample size?

In [37]:
def build_model(X, Y, kernel_func=None):
    variance = tf.math.reduce_variance(X)
    
    if kernel_func is None:
        kernel = gpflow.kernels.Matern52(variance=variance)
    else:
        kernel = kernel_func(variance)
        
    gpr = gpflow.models.GPR(
        data = (X, Y),
        kernel = gpflow.kernels.SquaredExponential(),
        likelihood = gpflow.likelihoods.Gaussian()
    )

    gpflow.utilities.print_summary(gpr, fmt="notebook")
    
    return GaussianProcessRegression(gpr)

In [38]:
# GP regression
model = build_model(X = X, Y = Y)

name,class,transform,prior,trainable,shape,dtype,value
GPR.kernel.variance,Parameter,Softplus,,True,(),float64,1
GPR.kernel.lengthscales,Parameter,Softplus,,True,(),float64,1
GPR.likelihood.variance,Parameter,Softplus + Shift,,True,(),float64,1


### Steps 4-6: Bayesian optimization loop

In [39]:
# create a dataset that works well with trieste
initial_data = trieste.data.Dataset(query_points=X, observations=Y)

In [40]:
# create the search space using trieste Box function
search_space = Box([-20, -20, -20, -20, -20, 0], [20, 20, 20, 20, 20, 1000])

In [41]:
search_space

Box(<tf.Tensor: shape=(6,), dtype=float64, numpy=array([-20., -20., -20., -20., -20.,   0.])>, <tf.Tensor: shape=(6,), dtype=float64, numpy=array([  20.,   20.,   20.,   20.,   20., 1000.])>, [], 1e-07)

In [42]:
ask_tell = trieste.ask_tell_optimization.AskTellOptimizer(search_space = search_space,
                                                          datasets = initial_data, 
                                                          models = model)

In [43]:
results = ask_tell.ask()

In [44]:
results

<tf.Tensor: shape=(1, 6), dtype=float64, numpy=
array([[ 16.73168087,  -7.17084516, -18.25408491, -18.64442522,
          7.76775255, 905.87565643]])>

In [45]:
def format_boundaries_after_ask(result):
    upper_bounds = np.array(result[0][0:3])
    lower_bounds = np.concatenate((result[0][3:5], [result[0][2]]))
    n_patients = np.array(result[0][5])

    return [
        upper_bounds,
        lower_bounds,
        n_patients
    ]

In [46]:
output = format_boundaries_after_ask(results)

In [47]:
output

[array([ 16.73168087,  -7.17084516, -18.25408491]),
 array([-18.64442522,   7.76775255, -18.25408491]),
 array(905.87565643)]

In [48]:
sim_results, alpha_prime, beta_prime, ess = simulate_group_sequential_designs(
    n_analyses=num_analyses,
    upper_bounds=output[0],
    lower_bounds=output[1],
    n_patients=output[2],
    alt_hypothesis=important_diff_delta,
    variance=assumed_variance
)

In [49]:
n_power09, beta_prime = find_sample_size(n_analyses=num_analyses,
                                         upper_bounds=output[0],
                                         lower_bounds=output[1],
                                         alt_hypothesis=important_diff_delta,
                                         variance=assumed_variance)

In [50]:
# 1. Find the number of patients that achieves 90% power (beta 0.1)
# here we get beta_prime and alpha prime
n_power09, beta_prime = find_sample_size(n_analyses=num_analyses,
                                         upper_bounds=output[0],
                                         lower_bounds=output[1],
                                         alt_hypothesis=important_diff_delta,
                                         variance=assumed_variance)

# 2. Generate the GPR input values
# note that the input includes the sample size at power 0.9
x3 = generate_gpr_input(n_analyses = num_analyses,
                       upper_bounds=output[0],
                       lower_bounds=output[1],
                       n_patients=n_power09)

# 3. Generate maximum expected sample size and feasibility penalty
max_ess_new = max_ess(n_analyses=num_analyses,
                      upper_bounds=output[0],
                      lower_bounds=output[1],
                      n_patients=n_power09)

penalty = feasibility_penalty(ratio=group_ratio,
                              variance=assumed_variance,
                              power=target_power,
                              alpha=target_alpha,
                              delta=important_diff_delta,
                              beta_prime=beta_prime,
                              alpha_prime=alpha_prime)


# 4. Calculate the function value (GPR output)
y3 = function_to_minimize(max_ess_val=max_ess_new, penalty=penalty)

In [51]:
x3

array([ 16.73168087,  -7.17084516, -18.25408491, -18.64442522,
         7.76775255, 245.66971352])

In [52]:
y3

245.6697135225238